<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [1]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00


In [2]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 10
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
BASE_FOLDER = 'drive/MyDrive/NLP_proj/estimation/'
OVERWRITE = ['']
MAX_RETRY = 10

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': "chatml"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=False,
                            force_download=True,
                            enable_thinking=True)


def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]



prompts = {}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [15]:
import re
test1 = 'mechanics: [A, B, ..., Z]'
test2 = 'a\nbhst\nOverall Complexity: 4.2'
test3 ='tsr\noptimal number of pLayers: 10'
test4 ='tsr\noptimal number of Player: 1'
test_dur_1 = 'sthser\nduration: 10-20'
test_dur_2 = 'sthser\nduration: 120'

regex = {'mechanics': r"(?i)^.*\bmechanics\s*:\s*\[.*\]\s*$",
         'complexity': r'(?i)^.*\bcomplexity:\s*[1-5]\.\d\s*$',
         'player': r'(?i)^.*\bplayer.*:\s*\d+\s*$',
         'duration': r'(?i)^.*\bduration.*:\s*(\d+)(?:-(\d+))?'

}
def check_output(prompt, output):
    if(prompt == 'all'):
        try:
            out_dic = json.loads(output)
            if('answer' in out_dic):
                return True
            else:
                return False
        except:
            return False
    else:
        last_line = output.split('\n')[-1]
        return bool(re.match(regex[prompt], last_line))


print(check_output('mechanics',test1))
print(check_output('complexity',test2))
print(check_output('player',test3))
print(check_output('player',test4))
print(check_output('duration',test_dur_1))
print(check_output('duration',test_dur_2))


True
True
True
True
True
True


# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [5]:

prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1.0‑5.0).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

<<<JSON schema>>>
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}


<<<EXAMPLE>>>
{
  "reasoning": "The rules describe moving pieces on a grid, controlling regions, and a simple scoring system. These map to Area Control and Hand Management. The rule density is low and decisions are straightforward, so complexity is around 1.8. With only 30 tokens and a small board, the game works best with 2‑3 players; 2 is optimal for maximum interaction. Each turn shifts control of a few squares, and a full game finishes in roughly 30 minutes.",
  "answer": {
    "mechanics": ["Area Control", "Hand Management"],
    "complexity": 1.8,
    "optimal player count": 2,
    "duration": 30
  }
}
"""

all_rf = {"type": "json_object",
          "schema": {
              "type": "object",
              "properties": {
                  "reasoning": {"type": "string"},
                  "answer": {"type": "object", "properties": {
                      "mechanics": {"type": "array", "items": {"type": "string"}},
                      "complexity": {"type": "number", "minimum": 1, "maximum": 5},
                      "optimal player count": {"type": "number"},
                      "duration": {"type": "number"},
                      "required": ["mechanics", "complexity", "optimal player count", "duration"]
                      }
                    }
                  },
              "required": ["reasoning", "answer"]
              }
          }



## Estimating each parameter separately

### Mechanics

In [6]:
prompts['mechanics'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s). Use only mechanics present in the list below.

here is a complete list of the available BGG mechanics:
[Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction,
"Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution,
Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events,
Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management,
Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato,
"I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction,
Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering,
Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control]


**Final Output**
mechanics: [A, B, ..., Z]
"""


### Complexity rating

In [7]:
prompts['complexity'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

**Final Output**
Learning: X
Playing: Y
Strategy: Z
Overall Complexity: W.W"""

### Optimal player count

In [8]:
prompts['player'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count based on your observations

**Final Output**
Optimal player count: X
"""

### Game duration

In [9]:
prompts['duration'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

**Final Output**
Average game duration: X minutes
"""

## Execution

In [ ]:
# select only prompt not executed
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                               or f'{CHOSEN}_{g}.json' in OVERWRITE]
if to_do == []:
    print('Nothing to do')

for g in to_do:
    print(g)
    output_dict = {p: {it: '' for it in range(IT)} for p in prompts.keys()}
    for p,it in tqdm([(p,it) for p in prompts.keys() for it in range(IT)]):
        for retry in range(MAX_RETRY):
            if(retry > 0 and VERBOSE):
                print(f'Retry {retry}')
            rulebook = requests.get(BASE_URL +g+'.txt').text
            name = g.replace('_',' ')
            out = model.create_chat_completion(generate_message(prompts[p], f'Here is the full rulebook of the game {name}:\n'+rulebook),
                                               temperature=models[CHOSEN]['temperature'],
                                               response_format = all_rf if p == 'all' else None,
                                               )['choices'][0]['message']['content']
            if(check_output(p,out)):
                break
        output_dict[p][it] = out
        if(VERBOSE):
            print(p,'-',it)
            print(out)

    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

ticket_to_ride


  2%|▏         | 1/50 [00:13<10:37, 13.01s/it]

all - 0
{ "reasoning": "The game involves building train routes, drawing train cards, and claiming tickets. The key actions are: drawing train cards, claiming routes, and drawing tickets. These map to Train Route Building, Resource Management, and Hand Management. The rule density is moderate, with multiple options and interactions. Decisions are strategic, but the game's simplicity keeps complexity at 2.5. The number of components suggests a 2-4 player count, but the game is best suited for 2-3 players. Each turn involves drawing cards, claiming routes, or drawing tickets, with the goal of completing routes and tickets. The game's average duration is around 30-45 minutes." , "answer": { "mechanics": ["Train Route Building", "Resource Management", "Hand Management"], "complexity": 2.5, "optimal player count": 2, "duration": 30 } }
Retry 1


  4%|▍         | 2/50 [00:40<17:12, 21.52s/it]

all - 1
{ "reasoning": "The rules describe drawing train cards, claiming routes, and drawing tickets. The game involves strategic planning, route management, and resource allocation. Key actions include drawing train cards to claim routes, claiming routes to score points, and drawing tickets to gain additional challenges or opportunities. The game's mechanics map to Hand Management, Route Building, and Ticket Collection. The rule density is moderate, with a good balance of complexity and clarity. The decision depth is around 3.2, considering the player's need to balance route management, ticket management, and train card management. With 110 train cards, 33 tickets, and a large game board, the game works best with 2-5 players. The optimal player count is 3, allowing for a good balance of interaction and individual strategy. Each turn involves drawing cards, claiming routes, or drawing tickets, and the game's average duration is around 60 minutes, considering the complexity of the requi

  6%|▌         | 3/50 [00:52<13:29, 17.22s/it]

all - 2
{ "reasoning": "The rules describe building train routes on a map, claiming routes, and managing tickets. Key actions include claiming routes, drawing train cards, and drawing tickets. These map to Route Building, Hand Management, and Ticket Management. The rule density is moderate, with decisions requiring some planning and strategy, so complexity is around 3.5. With many train cards, a large board, and multiple player interactions, the game works best with 2-5 players; 3 is optimal for maximum interaction. Each turn involves drawing cards, claiming routes, or drawing tickets, and a full game finishes in roughly 45 minutes." , "answer": { "mechanics": ["Route Building", "Hand Management", "Ticket Management"], "complexity": 3.5, "optimal player count": 3, "duration": 45 } }
